# TensoScan: entrenamiento del detector de dígitos en Colab

Ejecuta `synth → train → export → verify` con GPU. Antes de empezar: **Entorno de ejecución → Cambiar tipo → GPU**.

Los datos y los resultados se guardan en tu Drive (`TENSOSCAN_ML_DATA_DIR`), nunca en git. Si el repositorio es privado,
guarda un token de GitHub como secreto de Colab llamado `GITHUB_TOKEN` (icono de la llave).

## 1. Drive, GPU y repositorio

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/tensoscan-ml-data"  # cambia la ruta si quieres otra carpeta
os.makedirs(DATA_DIR, exist_ok=True)
os.environ["TENSOSCAN_ML_DATA_DIR"] = DATA_DIR
print("Datos en", DATA_DIR)

In [ ]:
!nvidia-smi -L || echo "SIN GPU: el entrenamiento completo tardaria mucho (cambia el tipo de entorno)"

In [ ]:
REPO = "TensoScan_Backend"  # nombre del repositorio en tu cuenta de GitHub
OWNER = "TU_USUARIO"        # <- ajusta
BRANCH = "recognition-refactor-feature"

try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{OWNER}/{REPO}.git"
except Exception:  # repositorio publico o sin secreto
    url = f"https://github.com/{OWNER}/{REPO}.git"

!rm -rf /content/{REPO}
!git clone --depth 1 --branch {BRANCH} {url} /content/{REPO}
%cd /content/{REPO}/ml

## 2. Instalación

`uv` instala PyTorch (con CUDA), ONNX, onnx2tf, TensorFlow y LiteRT en un entorno aislado (`.venv`); todos los comandos
siguientes usan `uv run`. Tarda unos minutos.

In [ ]:
!pip -q install uv
!uv sync --group dev --extra train --extra export
!uv run tensoscan-ml --version
!uv run tensoscan-ml audit-licenses --quiet

## 3. Datos sintéticos

30.000 imágenes de entrenamiento y 3.000 de validación (unos 15–25 min con 2 CPU). Si ya existen en Drive no se regeneran.
La semilla fija el resultado: la misma semilla da los mismos ficheros.

In [ ]:
SYNTH = f"{DATA_DIR}/synth/ds-synth-0.1.0"
if os.path.isdir(f"{SYNTH}/train/images"):
    print("Ya existe", SYNTH)
else:
    !uv run tensoscan-ml synth --out {SYNTH} --seed 42 --workers 2

In [ ]:
# Revisa visualmente unas muestras con sus etiquetas
!uv run tensoscan-ml synth --gallery /content/synth-samples.png
from IPython.display import Image

Image("/content/synth-samples.png")

## 4. Entrenamiento

`configs/train/yolox-nano-v1.yaml`: etapa 1 sintética; la etapa 2 (*fine-tuning* con fotogramas reales con cajas) se **omite
con un aviso** mientras no existan `data.real.train` y `data.real.val`. Con GPU, 1–3 h.

In [ ]:
!uv run tensoscan-ml train --config configs/train/yolox-nano-v1.yaml

In [ ]:
import glob

from IPython.display import Image

runs = sorted(glob.glob(f"{DATA_DIR}/runs/yolox-nano-v1"))
print(runs)
Image(f"{runs[-1]}/curves.png")

## 5. (Opcional) Etapa 2 con datos reales

1. Sube tus fotos/vídeos y el `labels.csv` a Drive (`REAL_SRC`) siguiendo `ml/docs/capture-protocol.md`.
2. Ejecuta `ingest` y `split`; pre-anota y corrige en CVAT (`ml/docs/cvat-annotation-guide.md`).
3. Apunta `data.real.train` y `data.real.val` de la configuración a los manifiestos con cajas y vuelve a lanzar `train`.

In [ ]:
REAL_SRC = f"{DATA_DIR}/real-source"  # carpeta con las fotos y labels.csv
if os.path.isdir(REAL_SRC):
    !uv run tensoscan-ml ingest --src {REAL_SRC} --dataset-version ds-real-0.1.0 --roi-mode center --preview
    # Mira {DATA_DIR}/real/ds-real-0.1.0/preview.jpg. Si el encuadre es correcto, procesa de verdad:
    # !uv run tensoscan-ml ingest --src {REAL_SRC} --dataset-version ds-real-0.1.0 --roi-mode center --commit-manifest
    # !uv run tensoscan-ml split --manifest {DATA_DIR}/real/ds-real-0.1.0/manifest.jsonl
else:
    print("No hay datos reales todavia:", REAL_SRC)

## 6. Exportar a `.tflite` y verificar

In [ ]:
import glob

stages = sorted(glob.glob(f"{DATA_DIR}/runs/yolox-nano-v1/*/best.pt"), key=os.path.getmtime)
CHECKPOINT = stages[-1]  # el mejor checkpoint de la ultima etapa ejecutada
VERSION = "0.1.0"
OUT = f"{DATA_DIR}/dist/v{VERSION}"
print("Checkpoint:", CHECKPOINT)
!uv run tensoscan-ml export --checkpoint {CHECKPOINT} --out {OUT} --version {VERSION} --dataset-version ds-synth-0.1.0

In [ ]:
# E/S del contrato, solo builtins, tamano, latencia en CPU y 20 imagenes doradas sinteticas
!uv run tensoscan-ml verify --model {OUT}/digits-{VERSION}.tflite --golden synthetic

In [ ]:
# El .tflite exportado (no el checkpoint) evaluado con el mismo pre/postprocesado que la app
!uv run tensoscan-ml eval --detector tflite --model {OUT}/digits-{VERSION}.tflite --manifest {SYNTH}/val/manifest.jsonl --out {OUT}/eval-synthetic
print(open(f"{OUT}/eval-synthetic/report.md").read()[:3000])

## 7. Resultado

En `DATA_DIR/dist/vX.Y.Z/` quedan `digits-X.Y.Z.tflite`, su `.json` (sidecar con el `sha256`), la plantilla `model-card.md`
(rellena los **TODO**) y los informes. Las métricas sintéticas **no cuentan para la aceptación**: el modelo solo se publica
si cumple los umbrales en el set de test **real** (`tensoscan-ml release`, tarea T-B003-13).

Descarga los ficheros de Drive o súbelos como *assets* de una GitHub Release; **no** los subas al repositorio.